# Final analysis: model quality and energy trade-offs

This notebook combines the validated and significant datasets, traditional models, and frozen CodeBERT results. It compares grouped out-of-fold predictions with the energy saved by recommendations, then reports break-even estimates only when both training and sampled inference have usable energy measurements. CodeBERT inference energy per pair is extrapolated from its 100-pair sample; the extrapolation is clearly labeled. Break-even uses only direct hardware-sensor or external power-meter readings, never CodeCarbon estimates. The 50 J subset is descriptive, not a significance test. An external power-meter template is created by Notebook 05 and is included when its readings have been completed.

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebooks 03-05 first to create the prepared datasets and model results.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from measurement_utils import EnergyCounter, process_manual_meter_entries

PROCESSED_DIR = ROOT / "Data" / "Processed"
RESULTS_ROOT = ROOT / "Results"
MEASUREMENT_ROOT = RESULTS_ROOT / "resource_measurements"
DATASET_NAMES = ("validated", "significant")
energy_counter = EnergyCounter()
print("Detected hardware energy counter:", energy_counter.backend or "unavailable")

if not MEASUREMENT_ROOT.is_dir():
    raise FileNotFoundError(f"No resource-measurement runs found under {MEASUREMENT_ROOT}.")
run_dirs = sorted(
    (path for path in MEASUREMENT_ROOT.iterdir() if path.is_dir()),
    key=lambda path: path.name,
    reverse=True,
)
tabular_required = [
    "run_metadata.json",
    *[
        f"{dataset}/{task}/{task}_repetitions.csv"
        for dataset in DATASET_NAMES
        for task in ("training", "inference")
    ],
]
codebert_required = [
    "run_metadata.json",
    "external_power_meter_manual_entry.csv",
    *[
        f"codebert/{dataset}/{task}_measurements.csv"
        for dataset in DATASET_NAMES
        for task in ("training", "inference")
    ],
    *[f"codebert/{dataset}/measurement_status.json" for dataset in DATASET_NAMES],
]
latest_tabular_run = next(
    (path for path in run_dirs if all((path / name).is_file() for name in tabular_required)),
    None,
)
latest_codebert_run = next(
    (path for path in run_dirs if all((path / name).is_file() for name in codebert_required)),
    None,
)
if latest_tabular_run is None:
    raise FileNotFoundError("No complete tabular resource run found; run Notebook 04 first.")
if latest_codebert_run is None:
    raise FileNotFoundError("No complete CodeBERT resource run found; run all cells in Notebook 05.")

performance_frames = []
prediction_frames = []
measurement_frames = []
for dataset_name in DATASET_NAMES:
    dataset_dir = RESULTS_ROOT / dataset_name
    for filename, model_family in (
        ("model_performance.csv", "traditional"),
        ("codebert_performance.csv", "codebert"),
    ):
        frame = pd.read_csv(dataset_dir / filename)
        frame["dataset"] = dataset_name
        frame["model_family"] = model_family
        performance_frames.append(frame)
    for filename, model_family in (
        ("model_oof_predictions.parquet", "traditional"),
        ("codebert_oof_predictions.parquet", "codebert"),
    ):
        frame = pd.read_parquet(dataset_dir / filename)
        frame["dataset"] = dataset_name
        frame["model_family"] = model_family
        prediction_frames.append(frame)

    for task_name in ("training", "inference"):
        frame = pd.read_csv(
            latest_tabular_run / dataset_name / task_name / f"{task_name}_repetitions.csv"
        )
        frame["dataset"] = dataset_name
        frame["model_family"] = "traditional"
        frame["resource_run_id"] = latest_tabular_run.name
        frame["actual_measurement"] = frame["actual_energy_measurement"].fillna(False).astype(bool)
        frame["backend"] = frame["energy_source"]
        frame["gross_energy_joules"] = frame["actual_energy_joules"]
        frame["net_energy_joules"] = frame["net_actual_energy_joules"]
        frame["n_inference_pairs"] = frame["n_unique_samples"] if task_name == "inference" else np.nan
        frame["n_inference_predictions"] = frame["n_predictions"] if task_name == "inference" else np.nan
        measurement_frames.append(frame)

        codebert_frame = pd.read_csv(
            latest_codebert_run / "codebert" / dataset_name / f"{task_name}_measurements.csv"
        )
        codebert_frame["dataset"] = dataset_name
        codebert_frame["model_family"] = "codebert"
        codebert_frame["resource_run_id"] = latest_codebert_run.name
        codebert_frame["feature_config"] = "codebert_embeddings"
        codebert_frame["actual_measurement"] = (
            codebert_frame["actual_energy_measurement"].fillna(False).astype(bool)
        )
        codebert_frame["backend"] = codebert_frame["energy_source"]
        codebert_frame["gross_energy_joules"] = codebert_frame["actual_energy_joules"]
        codebert_frame["net_energy_joules"] = codebert_frame["net_actual_energy_joules"]
        codebert_frame["n_inference_predictions"] = codebert_frame.get(
            "n_inference_predictions", codebert_frame.get("n_inference_pairs", np.nan)
        )
        measurement_frames.append(codebert_frame)

performance = pd.concat(performance_frames, ignore_index=True)
oof = pd.concat(prediction_frames, ignore_index=True)
measurements = pd.concat(measurement_frames, ignore_index=True, sort=False)
manual_meter_template = latest_codebert_run / "external_power_meter_manual_entry.csv"
manual_meter_results = latest_codebert_run / "external_power_meter_manual_results.csv"
if manual_meter_results.is_file():
    manual_meter = pd.read_csv(manual_meter_results)
else:
    manual_entries = pd.read_csv(manual_meter_template)
    required_meter_fields = [
        "initial_energy_reading",
        "final_energy_reading",
        "meter_unit",
        "measurement_duration_seconds",
        "experiment_name",
        "model_name",
        "task",
        "repetition",
    ]
    if not manual_entries.empty and manual_entries[required_meter_fields].notna().all().all():
        manual_meter = process_manual_meter_entries(
            manual_meter_template,
            manual_meter_results,
        )
    else:
        manual_meter = pd.DataFrame()
        print("External power-meter readings are not complete; the blank template is retained:", manual_meter_template)
if not manual_meter.empty:
    idle_meter = manual_meter.loc[manual_meter["task"].eq("idle")]
    idle_by_workload = {
        (row["dataset"], row["experiment_name"], int(row["repetition"])): row["actual_energy_joules"]
        for _, row in idle_meter.iterrows()
    }
    manual = manual_meter.loc[manual_meter["task"].isin(["training", "inference"])].copy()
    manual["idle_baseline_energy_joules"] = [
        idle_by_workload.get((row["dataset"], row["experiment_name"], int(row["repetition"])), np.nan)
        for _, row in manual.iterrows()
    ]
    if manual["idle_baseline_energy_joules"].isna().any():
        raise ValueError("Every manual training/inference reading needs its paired idle reading.")
    manual["model"] = "codebert_frozen_embeddings_logistic_regression"
    manual["feature_config"] = "codebert_embeddings"
    manual["model_family"] = "codebert_external_meter"
    manual["resource_run_id"] = latest_codebert_run.name
    manual["actual_measurement"] = True
    manual["backend"] = manual["energy_source"]
    manual["gross_energy_joules"] = manual["actual_energy_joules"]
    manual["net_energy_joules"] = (
        manual["actual_energy_joules"] - manual["idle_baseline_energy_joules"]
    )
    manual["elapsed_seconds"] = manual["measurement_duration_seconds"]
    manual["n_inference_pairs"] = np.where(manual["task"].eq("inference"), 100, np.nan)
    manual["n_inference_predictions"] = manual["n_inference_pairs"]
    measurements = pd.concat([measurements, manual], ignore_index=True, sort=False)

print("Tabular resource run:", latest_tabular_run.name)
print("CodeBERT resource run:", latest_codebert_run.name)
print("Measurements retain dataset, model family, and energy source separately.")

In [ ]:
model_energy_rows = []
large_subset_performance_rows = []
break_even_rows = []
large_difference_summaries = []

for dataset_name in DATASET_NAMES:
    results_dir = RESULTS_ROOT / dataset_name
    results_dir.mkdir(parents=True, exist_ok=True)
    prepared_path = PROCESSED_DIR / (
        "prepared_training_dataset.parquet"
        if dataset_name == "validated"
        else "prepared_significant_training_dataset.parquet"
    )
    prepared = pd.read_parquet(prepared_path)
    required_columns = {"problem_id", "result_energy_v0", "result_energy_v1"}
    missing_columns = sorted(required_columns - set(prepared.columns))
    if missing_columns:
        raise ValueError(f"{dataset_name} prepared data is missing columns: {missing_columns}")
    prepared["energy_diff_v0_minus_v1"] = prepared["result_energy_v0"] - prepared["result_energy_v1"]
    prepared["absolute_energy_difference_j"] = prepared["energy_diff_v0_minus_v1"].abs()
    large_pairs = prepared.loc[prepared["absolute_energy_difference_j"] > 50].copy()
    large_pairs["lower_energy_version"] = np.where(
        large_pairs["energy_diff_v0_minus_v1"] > 0, "V1", "V0"
    )
    large_pairs.to_csv(results_dir / "large_energy_difference_pairs.csv", index=False)
    large_difference_summaries.append({
        "dataset": dataset_name,
        "prepared_pairs": len(prepared),
        "pairs_abs_difference_over_50_j": len(large_pairs),
        "share_over_50_j": len(large_pairs) / len(prepared) if len(prepared) else np.nan,
        "mean_absolute_difference_over_50_j": large_pairs["absolute_energy_difference_j"].mean(),
        "median_absolute_difference_over_50_j": large_pairs["absolute_energy_difference_j"].median(),
        "v1_lower_energy_count_over_50_j": int((large_pairs["lower_energy_version"] == "V1").sum()),
        "v0_lower_energy_count_over_50_j": int((large_pairs["lower_energy_version"] == "V0").sum()),
    })

    dataset_oof = oof.loc[oof["dataset"].eq(dataset_name)].copy()
    dataset_oof["absolute_energy_difference_j"] = (
        dataset_oof["result_energy_v0"] - dataset_oof["result_energy_v1"]
    ).abs()
    large_oof = dataset_oof.loc[dataset_oof["absolute_energy_difference_j"] > 50]
    for (model_name, feature_config, model_family), predictions in dataset_oof.groupby(
        ["model", "feature_config", "model_family"]
    ):
        selected = predictions.loc[predictions["absolute_energy_difference_j"] > 50]
        if not selected.empty:
            truth = selected["y_true"].astype(int)
            predicted = selected["y_pred"].astype(int)
            probability = selected["y_probability"].astype(float)
            large_subset_performance_rows.append({
                "dataset": dataset_name,
                "model_family": model_family,
                "model": model_name,
                "feature_config": feature_config,
                "energy_difference_threshold_j": 50,
                "n_oof_pairs": len(selected),
                "accuracy": accuracy_score(truth, predicted),
                "balanced_accuracy": balanced_accuracy_score(truth, predicted),
                "f1": f1_score(truth, predicted, zero_division=0),
                "roc_auc": (
                    roc_auc_score(truth, probability)
                    if truth.nunique() == 2 and np.isfinite(probability).all()
                    else np.nan
                ),
            })

        chosen_energy = np.where(
            selected["y_pred"].to_numpy(dtype=int) == 1,
            selected["result_energy_v1"].to_numpy(),
            selected["result_energy_v0"].to_numpy(),
        )
        fixed_v0_energy = np.where(
            selected["was_swapped"].to_numpy(dtype=bool)
            if "was_swapped" in selected
            else np.zeros(len(selected), dtype=bool),
            selected["result_energy_v1"].to_numpy(),
            selected["result_energy_v0"].to_numpy(),
        )
        saving_per_pair = fixed_v0_energy - chosen_energy
        model_energy_rows.append({
            "dataset": dataset_name,
            "model_family": model_family,
            "model": model_name,
            "feature_config": feature_config,
            "oof_pairs_over_50_j": len(selected),
            "mean_recommended_energy_j": float(np.mean(chosen_energy)) if len(selected) else np.nan,
            "mean_saving_vs_always_v0_j": float(np.mean(saving_per_pair)) if len(selected) else np.nan,
            "total_saving_vs_always_v0_j": float(np.sum(saving_per_pair)) if len(selected) else np.nan,
        })

        matching = measurements.loc[
            measurements["dataset"].eq(dataset_name)
            & measurements["model"].eq(model_name)
            & measurements["feature_config"].eq(feature_config)
            & measurements["task"].isin(["training", "inference"])
        ].copy()
        sources = (
            matching["backend"].fillna("unavailable").unique().tolist()
            if not matching.empty
            else ["unavailable"]
        )
        for source in sources:
            source_rows = matching.loc[matching["backend"].fillna("unavailable").eq(source)]
            training = source_rows.loc[source_rows["task"].eq("training")]
            inference = source_rows.loc[source_rows["task"].eq("inference")]
            usable = (
                not training.empty
                and not inference.empty
                and training["actual_measurement"].fillna(False).astype(bool).all()
                and inference["actual_measurement"].fillna(False).astype(bool).all()
                and training["net_energy_joules"].notna().all()
                and inference["net_energy_joules"].notna().all()
                and inference["n_inference_predictions"].notna().all()
                and len(selected) > 0
            )
            if usable:
                training_energy = float(training["net_energy_joules"].mean())
                inference_energy = float(inference["net_energy_joules"].mean())
                prediction_count = float(inference["n_inference_predictions"].mean())
                inference_per_pair = inference_energy / prediction_count
                gross_saving = float(np.mean(saving_per_pair))
                net_saving = gross_saving - inference_per_pair
                break_even = math.ceil(training_energy / net_saving) if net_saving > 0 else None
                status = "finite" if break_even is not None else "No positive net saving per recommended pair."
            else:
                training_energy = inference_energy = inference_per_pair = np.nan
                prediction_count = float(inference["n_inference_predictions"].mean()) if not inference.empty and inference["n_inference_predictions"].notna().any() else np.nan
                gross_saving = float(np.mean(saving_per_pair)) if len(selected) else np.nan
                net_saving = np.nan
                break_even = None
                status = "Unavailable: actual energy for both training and inference is required."
            break_even_rows.append({
                "dataset": dataset_name,
                "model_family": model_family,
                "model": model_name,
                "feature_config": feature_config,
                "energy_source": source,
                "training_energy_j_per_full_fit": training_energy,
                "inference_energy_j_per_sampled_batch": inference_energy,
                "inference_predictions_per_batch": prediction_count,
                "inference_energy_j_per_pair": inference_per_pair,
                "mean_saving_vs_always_v0_j_per_pair_over_50_j": gross_saving,
                "net_saving_j_per_pair_after_inference": net_saving,
                "break_even_recommendations": break_even,
                "status": status,
            })

model_energy = pd.DataFrame(model_energy_rows)
large_subset_performance = pd.DataFrame(large_subset_performance_rows)
break_even_analysis = pd.DataFrame(break_even_rows)
large_difference_summary = pd.DataFrame(large_difference_summaries)
large_difference_summary.to_csv(RESULTS_ROOT / "large_energy_difference_summary.csv", index=False)
model_energy.to_csv(RESULTS_ROOT / "model_energy_impacts.csv", index=False)
large_subset_performance.to_csv(
    RESULTS_ROOT / "large_energy_difference_model_performance.csv", index=False
)
break_even_analysis.to_csv(RESULTS_ROOT / "break_even_analysis.csv", index=False)
display(large_difference_summary)
display(performance.merge(
    model_energy,
    on=["dataset", "model_family", "model", "feature_config"],
    how="left",
))
display(large_subset_performance)
display(break_even_analysis)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
for dataset_name in DATASET_NAMES:
    results_dir = RESULTS_ROOT / dataset_name
    dataset_performance = performance.loc[performance["dataset"].eq(dataset_name)]
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.barplot(
        data=dataset_performance,
        x="model",
        y="balanced_accuracy_mean",
        hue="model_family",
        ax=ax,
    )
    ax.set(title=f"{dataset_name.title()} data: grouped-CV balanced accuracy", xlabel="Model", ylabel="Mean balanced accuracy")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(results_dir / "final_model_balanced_accuracy.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    selected_measurements = measurements.loc[
        measurements["dataset"].eq(dataset_name)
        & measurements["task"].isin(["training", "inference"])
        & measurements["actual_measurement"].fillna(False).astype(bool)
        & measurements["net_energy_joules"].notna()
    ].copy()
    if selected_measurements.empty:
        print(f"{dataset_name}: no actual hardware or manual-meter energy readings are available.")
        continue
    training_energy = selected_measurements.loc[
        selected_measurements["task"].eq("training")
    ].groupby(
        ["model_family", "model", "feature_config"], as_index=False
    ).agg(training_energy_joules=("net_energy_joules", "mean"))
    inference_energy = selected_measurements.loc[
        selected_measurements["task"].eq("inference")
        & selected_measurements["n_inference_predictions"].gt(0)
    ].copy()
    inference_energy["inference_energy_joules_per_pair"] = (
        inference_energy["net_energy_joules"]
        / inference_energy["n_inference_predictions"]
    )
    inference_energy = inference_energy.groupby(
        ["model_family", "model", "feature_config"], as_index=False
    ).agg(inference_energy_joules_per_pair=("inference_energy_joules_per_pair", "mean"))
    energy_by_model = training_energy.merge(
        inference_energy,
        on=["model_family", "model", "feature_config"],
        how="inner",
    )
    energy_by_model["training_plus_one_inference_pair_joules"] = (
        energy_by_model["training_energy_joules"]
        + energy_by_model["inference_energy_joules_per_pair"]
    )
    performance_energy = dataset_performance.merge(
        energy_by_model,
        on=["model_family", "model", "feature_config"],
        how="inner",
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(
        data=performance_energy,
        x="balanced_accuracy_mean",
        y="training_plus_one_inference_pair_joules",
        hue="model_family",
        style="model",
        s=90,
        ax=ax,
    )
    ax.set(title=f"{dataset_name.title()} data: quality vs. measured compute energy", xlabel="Grouped-CV balanced accuracy", ylabel="Training + one recommendation inference (J)")
    fig.tight_layout()
    fig.savefig(results_dir / "model_performance_vs_energy.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

comparison_dir = RESULTS_ROOT / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
performance.to_csv(comparison_dir / "model_performance_comparison.csv", index=False)
energy_comparison = measurements.groupby(
    ["dataset", "model_family", "model", "feature_config", "task", "backend"],
    dropna=False,
    as_index=False,
).agg(
    resource_run_id=("resource_run_id", "first"),
    repetitions=("repetition", "nunique"),
    actual_measurement=("actual_measurement", "all"),
    mean_runtime_seconds=("elapsed_seconds", "mean"),
    mean_gross_energy_joules=("gross_energy_joules", "mean"),
    mean_net_energy_joules=("net_energy_joules", "mean"),
    mean_idle_baseline_energy_joules=("idle_baseline_energy_joules", "mean"),
    mean_codecarbon_estimated_energy_joules=("codecarbon_estimated_energy_joules", "mean"),
    mean_codecarbon_estimated_energy_joules_per_pair=("codecarbon_estimated_energy_joules_per_pair", "mean"),
    mean_extrapolated_codecarbon_energy_joules=("extrapolated_codecarbon_estimated_energy_joules", "mean"),
    extrapolation_target_pairs=("extrapolation_target_pairs", "mean"),
    inference_sample_seed=("inference_sample_seed", "first"),
    codecarbon_estimate_status=("codecarbon_estimate_status", "first"),
    mean_energy_joules_per_pair=("net_actual_energy_joules_per_pair", "mean"),
    mean_extrapolated_dataset_energy_joules=("extrapolated_net_actual_energy_joules", "mean"),
    inference_pairs=("n_inference_pairs", "mean"),
)
energy_comparison.to_csv(comparison_dir / "energy_measurement_comparison.csv", index=False)
display(energy_comparison)
print("Saved final analysis tables and figures under", RESULTS_ROOT)